<!--
Copyright (c) 2026 F. Hoffmann-La Roche Ltd
Authors:
- ERP Technology Platform Team <glospot_members@msxdl.roche.com>
-->

# Get the file names of the STRUST PSEs

Every PSE (Personal Security Environment) maintained in transaction `STRUST` is
stored as a file in the instance's `sec` directory, and in the database so that
all application servers share it. Inside SAP, the function module
`SSFPSE_FILENAME` (function group `SSFM`) returns that file name for a given
PSE, but it is **not remote-enabled**, so it cannot be called over RFC.

This notebook gets the same information on system **EH4** from remote-enabled
building blocks:

1. `SSFR_GET_ALL_STRUST_IDENTITIES` (function group `SSFR`) lists the PSEs
   shown in `STRUST`.
2. The naming rule of `SSFPSE_FILENAME`, ported to Python, turns each PSE into
   its file name.
3. `RFC_READ_TABLE` reads the customizing the rule needs (table `SSFARGS`) and
   the PSE files stored in the database (table `SSF_PSE_H`) to cross-check the
   result.

It does so with the two clients of `dragonrfc`:

- `dragonrfc.RFCClient`: the synchronous client, which blocks until the call returns.
- `dragonrfc.AsyncRFCClient`: the asyncio client, which runs the same call on a
  worker thread so the event loop stays free.

The logon uses **SNC with Kerberos**, so no password appears anywhere.

## Prerequisites

1. The SAP NW RFC SDK is installed and `dragonrfc` is built (`task setup`).
2. MIT Kerberos 5 is installed. Its GSSAPI library is the SNC library:
   `libgssapi_krb5.so.2` on Linux, `libgssapi_krb5.dylib` on macOS and
   `gssapi64.dll` on Windows.
3. You hold a valid Kerberos ticket for your user. Get one with
   `kinit prietoj4@EMEA.ROCHE.COM` and check it with `klist`.
4. Your user in EH4, client 010, is mapped to the SNC name
   `p:prietoj4@EMEA.ROCHE.COM` (transaction `SU01`, tab *SNC*) and has the
   authorizations to call `SSFR_GET_ALL_STRUST_IDENTITIES` and `RFC_READ_TABLE`
   over RFC (`S_RFC`) and to read the tables `TFDIR`, `SSFARGS` and `SSF_PSE_H`
   (`S_TABU_DIS` or `S_TABU_NAM`).

## Step 1: Logon parameters

`RFCClient` takes the SDK logon parameters as keyword arguments, all of them
`str`. All examples below share the same dictionary.

**Load-balanced logon through the message server**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `mshost` | `eh4scs.sap.roche.com` | Host of the message server (ASCS instance). |
| `msserv` | `3600` | Message server port (`36<instance number>`). |
| `sysid` | `EH4` | System ID; the message server serves only this system. |
| `group` | `SPACE` | Logon group (`SMLG`); the message server picks the least loaded application server in it. |
| `client` | `010` | SAP client. |

**SNC instead of user and password**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `snc_mode` | `1` | Turns SNC on. The SAP user is derived from the SNC name, so `user` and `passwd` are not set. |
| `snc_qop` | `9` | Quality of protection: `9` asks for the maximum the SNC library offers (integrity and privacy, i.e. encryption). |
| `snc_myname` | `p:prietoj4@EMEA.ROCHE.COM` | Your own SNC name: the Kerberos principal of your ticket. |
| `snc_partnername` | `p:sapEH4/EH4@EMEA.ROCHE.COM` | SNC name of the SAP system: the Kerberos service principal of EH4. |
| `snc_lib` | GSSAPI library | Path or name of the MIT Kerberos GSSAPI library. |

The cell finds the GSSAPI library with `ctypes.util.find_library`, which works on
every OS when MIT Kerberos is installed in the default location. If it does not
find it (for example, Homebrew's keg-only `krb5` on macOS), set the `SNC_LIB`
environment variable to the full path of the library before starting Jupyter.

In [2]:
import asyncio
import ctypes.util
import os

import dragonrfc

SNC_LIB = os.environ.get("SNC_LIB") or ctypes.util.find_library(
    "gssapi64" if os.name == "nt" else "gssapi_krb5",
)
assert SNC_LIB, "MIT Kerberos GSSAPI library not found: set SNC_LIB"

LOGON = {
    # Load balancing through the message server
    "mshost": "eh4scs.sap.roche.com",
    "msserv": "3600",
    "sysid": "EH4",
    "group": "SPACE",
    "client": "010",
    # SNC with Kerberos (GSSAPI)
    "snc_mode": "1",
    "snc_qop": "9",
    "snc_myname": "p:prietoj4@EMEA.ROCHE.COM",
    "snc_partnername": "p:sapEH4/EH4@EMEA.ROCHE.COM",
    "snc_lib": SNC_LIB,
}

print(f"dragonrfc {dragonrfc.__version__}, SDK {dragonrfc.sdk_version()}")
print(f"SNC library: {SNC_LIB}")

ModuleNotFoundError: No module named 'dragonrfc._core'

## Step 2: Check connectivity

Before calling any function module, make sure EH4 is reachable and accepts the
SNC logon. Opening the connection already performs the whole logon (message
server lookup, SNC handshake with your Kerberos ticket, user check);
`ping` then sends a round trip over the open connection.

Failures surface as typed exceptions, which the cell reports instead of a
traceback:

- `dragonrfc.RFCCommunicationError`: the message server or the application
  server cannot be reached (wrong host or port, firewall, VPN off), or the SNC
  library failed (for example, no valid Kerberos ticket).
- `dragonrfc.RFCLogonError`: EH4 rejected the logon (SNC name not mapped to a
  user, user locked, wrong client).

The same check is shown with both clients. Expected output: `EH4 reachable (sync)`
and `EH4 reachable (async)`. Continue only if both succeed.

In [ ]:
try:
    with dragonrfc.RFCClient(**LOGON) as conn:
        conn.ping()
except (dragonrfc.RFCCommunicationError, dragonrfc.RFCLogonError) as error:
    print(f"EH4 not reachable (sync): {type(error).__name__}: {error.message}")
else:
    print("EH4 reachable (sync)")

try:
    async with await dragonrfc.AsyncRFCClient.open(**LOGON) as aconn:
        await aconn.ping()
except (dragonrfc.RFCCommunicationError, dragonrfc.RFCLogonError) as error:
    print(f"EH4 not reachable (async): {type(error).__name__}: {error.message}")
else:
    print("EH4 reachable (async)")

## Step 3: Reading tables over RFC

Several steps below read database tables. The standard way is the
remote-enabled function module `RFC_READ_TABLE`: it takes the table name, the
fields to read (`FIELDS`) and an optional `WHERE` clause in Open SQL syntax
(`OPTIONS`, in lines of at most 72 characters), and returns each row as one
fixed-width text line in `DATA`. `FIELDS` comes back filled with the offset and
length of each field inside that line.

Two small helpers keep that out of the following steps:

- `table_query` builds the parameters of the call.
- `table_rows` cuts every `DATA` line into a `dict` keyed by field name.

They do not call SAP themselves, so the synchronous and the asynchronous
examples can share them.

In [ ]:
def table_query(table: str, fields: list[str], where: str = "") -> dict[str, object]:
    """Returns the RFC_READ_TABLE parameters to read fields of table."""
    return {
        "QUERY_TABLE": table,
        "FIELDS": [{"FIELDNAME": field} for field in fields],
        "OPTIONS": [{"TEXT": where}] if where else [],
    }


def table_rows(result: dict[str, object]) -> list[dict[str, str]]:
    """Returns the rows of an RFC_READ_TABLE result as dicts of stripped strings."""
    layout = [
        (field["FIELDNAME"], int(field["OFFSET"]), int(field["LENGTH"]))
        for field in result["FIELDS"]
    ]
    return [
        {
            name: row["WA"][offset : offset + length].strip()
            for name, offset, length in layout
        }
        for row in result["DATA"]
    ]

## Step 4: Why not call `SSFPSE_FILENAME` directly?

Only function modules flagged as *remote-enabled* can be called over RFC. The
flag is the processing type in transaction `SE37` (tab *Attributes*) and is
stored in the `FMODE` field of table `TFDIR`: `R` means remote-enabled.

The cell reads it for `SSFPSE_FILENAME` and for `SSFR_GET_ALL_STRUST_IDENTITIES`.
Expected output: `SSFPSE_FILENAME` is *not remote-enabled*,
`SSFR_GET_ALL_STRUST_IDENTITIES` is *remote-enabled*.

Calling `SSFPSE_FILENAME` anyway fails before the call even starts: to build the
call, the SDK reads the parameter types from the ABAP Dictionary, and the
`PSETEXT` exporting parameter has the generic type `CSEQUENCE`, which is not a
Dictionary type. Both `describe_function` and `call` then raise
`dragonrfc.RFCABAPApplicationError` with key `NOT_FOUND`.

In [ ]:
with dragonrfc.RFCClient(**LOGON) as conn:
    result = conn.call(
        "RFC_READ_TABLE",
        **table_query(
            "TFDIR",
            ["FUNCNAME", "FMODE"],
            "FUNCNAME IN ('SSFPSE_FILENAME', 'SSFR_GET_ALL_STRUST_IDENTITIES')",
        ),
    )

for row in table_rows(result):
    remote = "remote-enabled" if row["FMODE"] == "R" else "not remote-enabled"
    print(f"{row['FUNCNAME']:<31} {remote}")

## Step 5: List the PSEs of `STRUST` (synchronous)

`SSFR_GET_ALL_STRUST_IDENTITIES` returns the PSEs that `STRUST` shows in its
tree. It has no importing parameters. In the result:

- `ET_STRUST_IDENTITIES` holds one row per PSE: its context (`PSE_CONTEXT`), its
  application (`PSE_APPLIC`) and its description (`PSE_DESCRIPT`). Context and
  application together identify a PSE; they are the `CONTEXT` and `APPLIC`
  importing parameters of `SSFPSE_FILENAME`.
- `ET_BAPIRET2` holds messages. The function module reports errors there
  instead of raising exceptions, so the cell checks it for messages of type `E`
  (error) or `A` (abort).

`call` blocks the current thread until EH4 answers; it releases the GIL while
waiting, so other Python threads keep running. The `with` block closes the
connection at the end, even if an error is raised.

Expected output: one line per PSE with its context, application and description.

In [ ]:
with dragonrfc.RFCClient(**LOGON) as conn:
    result = conn.call("SSFR_GET_ALL_STRUST_IDENTITIES")

errors = [msg["MESSAGE"] for msg in result["ET_BAPIRET2"] if msg["TYPE"] in {"E", "A"}]
assert not errors, errors

identities = [
    (row["PSE_CONTEXT"], row["PSE_APPLIC"], row["PSE_DESCRIPT"])
    for row in result["ET_STRUST_IDENTITIES"]
]
for context, applic, descript in identities:
    print(f"{context:<4} {applic:<8} {descript}")

## Step 6: The naming rule of `SSFPSE_FILENAME`

`SSFPSE_FILENAME` derives the file name from the context and the application
(see its source code in `SE37`):

| Context | Application | File name |
|---------|-------------|-----------|
| `PROG` | `<SYST>` | `SAPSYS.pse` (System PSE) |
| `PROG` | `<SNCS>` | `SAPSNCS.pse` (SNC SAPCryptolib) |
| `PROG` | `<SSLS>` | `SAPSSLS.pse` (old name of the SSL server Standard PSE) |
| `PROG` | `<FILE>` | none: an arbitrary file chosen in `STRUST` |
| `SSLS` | `DFAULT` | `SAPSSLS.pse` |
| `SSLS` | other | `SAPSSLS_<applic>.pse` |
| `SSLC` | `ANONYM` | `SAPSSLA.pse` |
| `SSLC` | `DFAULT` | `SAPSSLC.pse` |
| `SSLC` | `<SPKI>` | `sap_system_pki_instance.pse` (System PKI, per instance) |
| `SSLC` | other | `SAPSSL<applic>.pse` |
| `WSSE` | `DFAULT` | `SAPWSSE.pse` |
| `WSSE` | other | `SAPWSSE_<applic>.pse` |
| `SMIM` | any | `SAPSMIM_<applic><client>.pse` |
| `SSFA` | any | the `PROFILE` field of table `SSFARGS` for that application (transaction `SSFA`) |

`pse_filename` implements that table in Python: `FIXED_NAMES` holds the
fixed names and `PATTERNS` the rule for any other application. `SSFA` PSEs are customized per
application, so their file names come from `SSFARGS`, passed in as a `dict`
from application to profile. For a PSE that is not distributed to all
application servers, `SSFPSE_FILENAME` returns the profile as a full path
instead of a file name; `pse_filename` returns it as well. It returns `None`
for `PROG/<FILE>` and for unknown contexts, where `SSFPSE_FILENAME` returns no
name or raises `PSE_NOT_FOUND`.

The `assert` lines check the function against rows of the table above.

In [ ]:
# Fixed file names, keyed by (context, application)
FIXED_NAMES = {
    ("PROG", "<SYST>"): "SAPSYS.pse",
    ("PROG", "<SNCS>"): "SAPSNCS.pse",
    ("PROG", "<SSLS>"): "SAPSSLS.pse",
    ("SSLS", "DFAULT"): "SAPSSLS.pse",
    ("SSLC", "ANONYM"): "SAPSSLA.pse",
    ("SSLC", "DFAULT"): "SAPSSLC.pse",
    ("SSLC", "<SPKI>"): "sap_system_pki_instance.pse",
    ("WSSE", "DFAULT"): "SAPWSSE.pse",
}
# File name patterns for the other applications of a context
PATTERNS = {
    "SSLS": "SAPSSLS_{applic}.pse",
    "SSLC": "SAPSSL{applic}.pse",
    "WSSE": "SAPWSSE_{applic}.pse",
    "SMIM": "SAPSMIM_{applic}{client}.pse",
}


def pse_filename(
    context: str,
    applic: str,
    ssfargs: dict[str, str],
    client: str = LOGON["client"],
) -> str | None:
    """Returns the file name of a STRUST PSE, as SSFPSE_FILENAME derives it."""
    if context == "SSFA":
        return ssfargs.get(applic)
    if (context, applic) in FIXED_NAMES:
        return FIXED_NAMES[context, applic]
    pattern = PATTERNS.get(context)
    return pattern.format(applic=applic, client=client) if pattern else None


assert pse_filename("SSLS", "DFAULT", {}) == "SAPSSLS.pse"
assert pse_filename("SSLC", "ANONYM", {}) == "SAPSSLA.pse"
assert pse_filename("WSSE", "WSSKEY", {}) == "SAPWSSE_WSSKEY.pse"
assert pse_filename("SMIM", "DFAULT", {}) == "SAPSMIM_DFAULT010.pse"
assert pse_filename("SSFA", "SSO2", {"SSO2": "SAPSYS.pse"}) == "SAPSYS.pse"
assert pse_filename("PROG", "<FILE>", {}) is None

## Step 7: File names and cross-check (synchronous)

This step puts the pieces together on one connection, one call after the other:

1. Read `APPLIC` and `PROFILE` of table `SSFARGS` for the `SSFA` PSEs.
2. Read the `FILENAME` field of table `SSF_PSE_H`, the header table of the PSE
   files stored in the database.
3. Print every PSE from step 5 with its file name, and whether that file is
   stored in the database.

A PSE whose file is not in `SSF_PSE_H` is listed in `STRUST` but was never
created in this system, or it is not distributed (the file exists on one
application server only).

`report` is a plain function, so the asynchronous step below reuses it.

Expected output: one line per PSE with its context, application and file name,
followed by `stored` or `not stored`.

In [ ]:
def report(
    identities: list[tuple[str, str, str]],
    ssfargs: dict[str, str],
    stored: set[str],
) -> None:
    """Prints the file name of every PSE and whether it is stored in the database."""
    for context, applic, _ in identities:
        filename = pse_filename(context, applic, ssfargs)
        status = "stored" if filename in stored else "not stored"
        print(f"{context:<4} {applic:<8} -> {filename or '-':<32} {status}")


with dragonrfc.RFCClient(**LOGON) as conn:
    ssfargs_result = conn.call(
        "RFC_READ_TABLE",
        **table_query("SSFARGS", ["APPLIC", "PROFILE"]),
    )
    pse_h_result = conn.call("RFC_READ_TABLE", **table_query("SSF_PSE_H", ["FILENAME"]))

ssfargs = {row["APPLIC"]: row["PROFILE"] for row in table_rows(ssfargs_result)}
stored = {row["FILENAME"] for row in table_rows(pse_h_result)}
report(identities, ssfargs, stored)

## Step 8: The same, asynchronously and concurrently

`AsyncRFCClient` has the same methods as `RFCClient`, but they are coroutines.
`AsyncRFCClient.open` opens the connection on a worker thread, so the logon
does not block the event loop either; `async with` closes it at the end.

The three calls of steps 5 and 7 do not depend on each other, so they can run
at the same time. One connection runs one call at a time (the SDK forbids
sharing a connection between threads), so `fetch` opens its own connection per
call, and `asyncio.gather` runs the three of them concurrently. If one fails,
`gather` raises its exception and the cell stops.

Jupyter runs an event loop already, so `await` works at the top level of a
cell. In a script, put the code in an `async def main()` function and run it
with `asyncio.run(main())`.

Expected output: the same lines as in step 7.

In [ ]:
async def fetch(name: str, **params: object) -> dict[str, object]:
    """Calls a function module on its own connection and returns the result."""
    async with await dragonrfc.AsyncRFCClient.open(**LOGON) as aconn:
        return await aconn.call(name, **params)


identities_result, ssfargs_result, pse_h_result = await asyncio.gather(
    fetch("SSFR_GET_ALL_STRUST_IDENTITIES"),
    fetch("RFC_READ_TABLE", **table_query("SSFARGS", ["APPLIC", "PROFILE"])),
    fetch("RFC_READ_TABLE", **table_query("SSF_PSE_H", ["FILENAME"])),
)

errors = [
    msg["MESSAGE"]
    for msg in identities_result["ET_BAPIRET2"]
    if msg["TYPE"] in {"E", "A"}
]
assert not errors, errors

identities = [
    (row["PSE_CONTEXT"], row["PSE_APPLIC"], row["PSE_DESCRIPT"])
    for row in identities_result["ET_STRUST_IDENTITIES"]
]
ssfargs = {row["APPLIC"]: row["PROFILE"] for row in table_rows(ssfargs_result)}
stored = {row["FILENAME"] for row in table_rows(pse_h_result)}
report(identities, ssfargs, stored)

## Summary

- `SSFPSE_FILENAME` is not remote-enabled. Its result can still be rebuilt over
  RFC: `SSFR_GET_ALL_STRUST_IDENTITIES` lists the PSEs, the naming rule maps
  them to file names, and `SSFARGS` and `SSF_PSE_H` (read with `RFC_READ_TABLE`)
  supply the customized names and the stored files.
- The naming rule is a copy of the ABAP code. If a future release changes
  `SSFPSE_FILENAME`, compare its source in `SE37` with `pse_filename`.
- Both clients take the same logon parameters and return the same results and
  errors; only the calling style differs.
- Use `RFCClient` in scripts, command-line tools and threaded code: it is the
  simplest choice and a `with` block guarantees the connection is closed.
- Use `AsyncRFCClient` inside asyncio applications (web services, bots,
  notebooks running many lookups): calls do not block the event loop, and
  separate connections run concurrently.
- SNC with Kerberos keeps credentials out of the code: the SAP user comes from
  your Kerberos ticket and QoP `9` encrypts the traffic.